In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd /content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/

/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능


In [ ]:
!pip install transformers
!pip install torchinfo
!pip install pytorch-lightning
!pip install torchmetrics
!pip install konlpy
!pip install soynlp
!pip install sentenecepiece
!pip install -q -U bitsandbytes
!pip install -q -U git+https://github.com/huggingface/transformers.git
!pip install -q -U git+https://github.com/huggingface/peft.git
!pip install -q -U git+https://github.com/huggingface/accelerate.git
!pip install -q -U datasets scipy ipywidgets
!pip install -q wandb -U

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 59.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.1/311.1 kB 36.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 108.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 91.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.0/295.0 kB 35.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 776.3/776.3 kB 13.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 805.2/805.2 kB 56.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 66.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 465.3/465.3 kB 49.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 416.8/416.8 kB 8.8 MB/s eta 0:00:00
ERROR: Could not find a version that satisfies the requirement sentenecepiece (from versions: none)
ERROR: No matching distribution found for sentenecepiece
     ━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import numpy as np
import pandas as pd
import os, sys, re, math, random, time, json, pickle, tqdm, gc
from collections import defaultdict
import itertools as it

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from transformers import AdamW, get_linear_schedule_with_warmup
from transformers.optimization import get_cosine_schedule_with_warmup
from transformers import AutoTokenizer, AutoModelForSequenceClassification

from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import f1_score, accuracy_score
from sklearn import preprocessing
from sklearn.model_selection import train_test_split
from tqdm import tqdm

class args:
    # amp = True
    gpu = '0'

    label_size = 2
    epochs=10
    batch_size=8
    weight_decay=1e-6
    max_len = 60

    start_lr = 2e-5

    num_workers=0
    seed=2022

os.environ["CUDA_VISIBLE_DEVICES"] = args.gpu
device = torch.device(f"cuda" if torch.cuda.is_available() else "cpu")
print(device)
##----------------
def set_seeds(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seeds(seed=args.seed)

path = '/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/'

df_train = pd.read_csv('/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/기말과제/df_train.csv', encoding = 'utf-8')
df_dev = pd.read_csv('/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/기말과제/df_dev.csv', encoding = 'utf-8')
df_test = pd.read_csv('/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/기말과제/df_test.csv', encoding = 'utf-8')

df_train.dropna(inplace = True)
df_dev.dropna(inplace = True)
df_test.dropna(inplace = True)

df_train.reset_index(drop = True, inplace = True)
df_dev.reset_index(drop = True, inplace = True)

cuda


In [ ]:
import torch
from transformers import BitsAndBytesConfig

base_model_id = "EleutherAI/polyglot-ko-12.8b"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

base_model = AutoModelForSequenceClassification.from_pretrained(base_model_id, quantization_config=bnb_config)

Loading checkpoint shards:   0%|          | 0/28 [00:00<?, ?it/s]

Some weights of GPTNeoXForSequenceClassification were not initialized from the model checkpoint at EleutherAI/polyglot-ko-12.8b and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    base_model_id,
    model_max_length=512,
    padding_side="left",
    add_eos_token=True)

tokenizer.eos_token = tokenizer.eos_token
tokenizer.pad_token = tokenizer.pad_token if tokenizer.pad_token is not None else tokenizer.eos_token

In [ ]:
gc.collect()
torch.cuda.empty_cache()

In [ ]:
from tokenizers.processors import TemplateProcessing

class CustomDataset(Dataset):
    def __init__(self, dataset, text, label, tokenizer, max_len):
        self.dataset = dataset
        self.text = text
        self.label = label
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __getitem__(self, idx):
        text = self.dataset.iloc[idx]['input']
        label = self.dataset.iloc[idx]['output']

        inputs = self.tokenizer(text,
                                return_tensors='pt',
                                truncation=True,
                                max_length=self.max_len,
                                padding='max_length',
                                add_special_tokens=True)

        input_ids = inputs['input_ids'][0]
        attention_mask = inputs['attention_mask'][0]

        return input_ids, attention_mask, label

    def __len__(self):
        return len(self.dataset)

def collate_batch(batch):
    input_ids = [item[0] for item in batch]
    attention_masks = [item[1] for item in batch]
    labels = torch.tensor([item[2] for item in batch])  # 레이블을 Tensor로 변환

    # 패딩 추가
    input_ids = torch.nn.utils.rnn.pad_sequence(input_ids, batch_first=True, padding_value=tokenizer.pad_token_id)
    attention_masks = torch.nn.utils.rnn.pad_sequence(attention_masks, batch_first=True, padding_value=0)

    return input_ids, attention_masks, labels

train_dataset = CustomDataset(df_train, 'input', 'output', tokenizer, args.max_len)
train_loader = DataLoader(train_dataset, batch_size=args.batch_size, shuffle=True, collate_fn=collate_batch)
valid_dataset = CustomDataset(df_dev, 'input', 'output', tokenizer, args.max_len)
valid_loader = DataLoader(valid_dataset, batch_size=args.batch_size, shuffle=False, collate_fn=collate_batch)

In [ ]:
from peft import get_peft_config, get_peft_model, LoraConfig, TaskType, prepare_model_for_kbit_training

peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS, inference_mode=False, r=8, lora_alpha=16, lora_dropout=0.05, bias="none"
)

model = prepare_model_for_kbit_training(base_model)
peft_model = get_peft_model(model, peft_config)

In [ ]:
# GPU 11.6GB 사용
import transformers
from datetime import datetime
import torch.optim as optim
import torch.nn.functional as F

peft_model.config.num_labels = 2
peft_model.config.pad_token_id = tokenizer.pad_token_id

loss_fn = nn.CrossEntropyLoss()
optimizer = AdamW(peft_model.parameters(), lr=args.start_lr)
scheduler = get_linear_schedule_with_warmup(optimizer,
                                            num_warmup_steps = int(len(train_loader)*args.epochs * 0.1),
                                            num_training_steps = len(train_loader)*args.epochs) # cosine annealing
# micro f1으로 바꾸기
def train(peft_model, data_loader, loss_fn, optimizer, scheduler):
    train_loss = 0
    target_lst = []
    pred_lst = []
    i = 0
    peft_model.train()
    pbar = tqdm(data_loader)
    for ids, mask, target in pbar:
        i += 1
        ids = ids.to(device)
        mask = mask.to(device)
        target = target.to(device)

        optimizer.zero_grad()
        output = peft_model(ids, mask)
        loss = loss_fn(output.logits, target).to(device)
        train_loss += loss.item()

        loss.backward()
        optimizer.step()
        scheduler.step()

        target_lst.extend(target.detach().cpu().numpy())
        pred_lst.extend(output.logits.argmax(dim=1).tolist())
        pbar.set_description('\033[1m[C_loss : {:>.5}]\033[0m'.format(round(train_loss / i, 4)))
    train_loss = train_loss / len(data_loader)
    train_score = f1_score(y_true=target_lst, y_pred=pred_lst, average='macro')
    train_acc = accuracy_score(y_true=target_lst, y_pred=pred_lst)

    torch.cuda.empty_cache()
    gc.collect()

    return peft_model, train_loss, train_score, train_acc

def valid(peft_model, data_loader, loss_fn):
    valid_loss = 0
    target_lst = []
    pred_lst = []
    i = 0
    peft_model.eval()
    pbar = tqdm(data_loader)
    for ids, mask, target in pbar:
        i += 1
        ids = ids.to(device)
        mask = mask.to(device)
        target = target.to(device)

        output = peft_model(ids, mask)
        loss = loss_fn(output.logits, target).to(device)
        valid_loss += loss.item()

        target_lst.extend(target.detach().cpu().numpy())
        pred_lst.extend(output.logits.argmax(dim=1).tolist())
        pbar.set_description('\033[1m[C_loss : {:>.5}]\033[0m'.format(round(valid_loss / i, 4)))
    valid_loss = valid_loss / len(data_loader)
    valid_score = f1_score(y_true=target_lst, y_pred=pred_lst, average='macro')
    valid_acc = accuracy_score(y_true=target_lst, y_pred=pred_lst)

    torch.cuda.empty_cache()
    gc.collect()

    return peft_model, valid_loss, valid_score, valid_acc

stop_val_f1 = 0
stop_count = 0

for epoch in range(args.epochs):
    if stop_count == 3:
        break

    peft_model, train_loss, train_score, train_acc = train(peft_model, train_loader, loss_fn, optimizer, scheduler)
    peft_model, valid_loss, valid_score, valid_acc = valid(peft_model, valid_loader, loss_fn)

    print(f'Epoch : {epoch + 1},    t_loss : {round(train_loss, 4)},   t_f1 : {round(train_score, 4)},   t_acc : {round(train_acc, 4)}\n')
    print(f'              v_loss : {round(valid_loss, 4)},   v_f1 : {round(valid_score, 4)},   v_acc : {round(valid_acc, 4)}\n')

    if valid_score > stop_val_f1:
        default_weight_path = path + 'ddd_polyglot_ko_12.8b_t_e.pt'
        torch.save(peft_model.state_dict(), default_weight_path)
        stop_val_f1 = valid_score
        stop_count = 0
    else:
        stop_count += 1

    torch.cuda.empty_cache()
    gc.collect()

/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:423: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
[C_loss : 0.2382]: 100%|██████████| 260/260 [00:59<00:00,  4.33it/s]


Epoch : 1,    t_loss : 0.701,   t_f1 : 0.7287,   t_acc : 0.7318

              v_loss : 0.2382,   v_f1 : 0.9131,   v_acc : 0.9151



[C_loss : 0.158]: 100%|██████████| 260/260 [00:59<00:00,  4.35it/s]


Epoch : 2,    t_loss : 0.174,   t_f1 : 0.9333,   t_acc : 0.9357

              v_loss : 0.158,   v_f1 : 0.9472,   v_acc : 0.9489



[C_loss : 0.1487]: 100%|██████████| 260/260 [00:59<00:00,  4.36it/s]


...

Epoch : 4,    t_loss : 0.0807,   t_f1 : 0.9688,   t_acc : 0.9699

              v_loss : 0.1503,   v_f1 : 0.949,   v_acc : 0.9508



[C_loss : 0.173]: 100%|██████████| 260/260 [00:59<00:00,  4.35it/s]


Epoch : 5,    t_loss : 0.05,   t_f1 : 0.983,   t_acc : 0.9835

              v_loss : 0.173,   v_f1 : 0.9472,   v_acc : 0.9489



[C_loss : 0.2208]: 100%|██████████| 260/260 [00:59<00:00,  4.34it/s]


Epoch : 6,    t_loss : 0.0262,   t_f1 : 0.9925,   t_acc : 0.9928

              v_loss : 0.2208,   v_f1 : 0.9411,   v_acc : 0.9426



In [ ]:
'''
        log_probs = F.log_softmax(outputs.logits, dim=-1)
        print(f'Original log_probs shape: {log_probs.shape}')
        Original log_probs shape: torch.Size([32, 2])
'''

In [ ]:
import torch
from transformers import BitsAndBytesConfig

base_model_id = "EleutherAI/polyglot-ko-12.8b"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

base_model = AutoModelForSequenceClassification.from_pretrained(base_model_id, quantization_config=bnb_config)

In [ ]:
class TestDataset(Dataset):

    def __init__(self, dataset, text, tokenizer, max_len):
        self.dataset = dataset
        self.text = text
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __getitem__(self, idx):
        text = self.dataset.iloc[idx]['input']
        inputs = self.tokenizer(text,
                                return_tensors='pt',
                                truncation=True,
                                max_length=self.max_len,
                                padding='max_length',
                                add_special_tokens=True)

        input_ids = inputs['input_ids'][0]
        attention_mask = inputs['attention_mask'][0]

        return input_ids, attention_mask

    def __len__(self):
        return len(self.dataset)

def collate_batch(batch):
    input_ids = [item[0] for item in batch]
    attention_masks = [item[1] for item in batch]

    # 패딩 추가
    input_ids = torch.nn.utils.rnn.pad_sequence(input_ids, batch_first=True, padding_value=tokenizer.pad_token_id)
    attention_masks = torch.nn.utils.rnn.pad_sequence(attention_masks, batch_first=True, padding_value=0)

    return input_ids, attention_masks

tokenizer = AutoTokenizer.from_pretrained(
    base_model_id,
    model_max_length=args.max_len,
    add_eos_token=True)
tokenizer.pad_token = tokenizer.eos_token
test_dataset = TestDataset(df_test, 'input', tokenizer, args.max_len)
test_loader = DataLoader(test_dataset, batch_size=args.batch_size, shuffle=False, collate_fn=collate_batch)

In [ ]:
from peft import get_peft_config, get_peft_model, LoraConfig, TaskType, prepare_model_for_kbit_training

peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS, inference_mode=False, r=8, lora_alpha=16, lora_dropout=0.05, bias="none"
)

model = prepare_model_for_kbit_training(base_model)
peft_model = get_peft_model(model, peft_config)

In [ ]:
peft_model.load_state_dict(torch.load('/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/ddd_polyglot_ko_12.8b_t_e.pt'))
peft_model.to(device)

peft_model.eval()
res = []
with torch.no_grad():
    for t, data in enumerate(tqdm(test_loader)):
        ids = data[0].to(device)
        mask = data[1].to(device)

        output = peft_model(ids, mask)[0]
        output = output.cpu().numpy()
        res.extend(output.argmax(axis = 1).tolist())

df_test['output'] = res

Loading checkpoint shards:   0%|          | 0/28 [00:00<?, ?it/s]

Some weights of GPTNeoXForSequenceClassification were not initialized from the model checkpoint at EleutherAI/polyglot-ko-12.8b and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
100%|██████████| 259/259 [00:57<00:00,  4.48it/s]


In [ ]:
df_test

,Unnamed: 0,id,input,output
0,0,nikluge-au-2022-test-000001,극좌는 이 비겁자층을 제대로 요리할 줄 안다...,0
1,1,nikluge-au-2022-test-000002,내가 진짜 올 해 안에 차 산다!,0
2,2,nikluge-au-2022-test-000003,"선거 때마다 불장난 하는 못된 버릇 대대손손 배워가지고 그러고 까불어대면, 너 나중...",1
3,3,nikluge-au-2022-test-000004,"난 99년도에 이미 세상은 망해서 선한자들은 이미 모두 하늘로 올라갔고, 남은 우리...",1
4,4,nikluge-au-2022-test-000005,피의자로 가는 싸가지 없는 쓰래기!,1
...,...,...,...,...
2067,2067,nikluge-au-2022-test-002068,근데 비계 올 사람만 찍어,0
2068,2068,nikluge-au-2022-test-002069,지들 입맛대로 막 가네 미친,1
2069,2069,nikluge-au-2022-test-002070,얘는 지가 이걸 쓰면서 왜 이런 생각을 못하지?,0
2070,2070,nikluge-au-2022-test-002071,알겟나요,0


In [ ]:
def jsonlload(fname):
    with open(fname, "r", encoding="utf-8") as f:
        lines = f.read().strip().split("\n")
        j_list = [json.loads(line) for line in lines]
    return j_list

test_df = jsonlload('/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/기말과제/NIKL_AU_2023_COMPETITION_v1.0/nikluge-au-2022-test.jsonl')
len(test_df)

2072

In [ ]:
for i, p in enumerate(df_test['output']):
    test_df[i]['output'] = p

In [ ]:
def jsonldump(j_list, fname):
    with open(fname, "w", encoding='utf-8') as f:
        for json_data in j_list:
            f.write(json.dumps(json_data, ensure_ascii=False)+'\n')
jsonldump(test_df, '/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/test5_polyglot_ko_12.8b.jsonl')